# Failure Analysis

Use this notebook to inspect low-quality, failed, or surprising evaluation results. It focuses on Postgres eval data and complements runtime investigation in `docs/analytics/observability-evaluation-workflow.md`.

Typical flow:

1. Load recent `eval_runs` and `eval_samples`.
2. Select failed, warning, or low-scoring runs.
3. Inspect sample-level inputs, outputs, references, and task-specific details.
4. Group failures into actionable categories.
5. Export candidate rows for future RAG eval datasets.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

from shared.config import get_settings, secret_value

pd.set_option("display.max_colwidth", 240)
pd.set_option("display.max_columns", 80)

settings = get_settings()
db_url = secret_value(settings.eval.db_url) or secret_value(settings.platform.postgres_dsn)
engine = create_engine(db_url)
exports_dir = Path("../../artifacts/eval/failure_analysis").resolve()
exports_dir.mkdir(parents=True, exist_ok=True)
exports_dir

## Recent Runs

In [ ]:
recent_runs_sql = text("""
select
    id,
    created_at,
    finished_at,
    status,
    task,
    dataset_name,
    metric_name,
    metric_value,
    eval_verdict,
    base_model,
    lora_alias,
    rag_enabled,
    knowledge_base,
    rag_alias,
    qdrant_alias,
    qdrant_collection,
    rag_manifest_id,
    extra,
    error_message
from eval_runs
order by created_at desc
limit 200
""")

runs = pd.read_sql(recent_runs_sql, engine)
runs.head(20)

## Runs That Need Attention

In [ ]:
attention = runs[
    (runs["status"] != "completed") | (runs["eval_verdict"].isin(["warn", "fail"]))
].copy()

attention.sort_values(["created_at", "task", "dataset_name"], ascending=[False, True, True]).head(
    50
)

If thresholds are not configured yet, use metric-specific cutoffs manually. Adjust the values below for the current experiment.

In [ ]:
manual_cutoffs = {
    "rouge_l": 0.15,
    "bertscore_f1": 0.50,
    "pass_at_1": 0.30,
    "recall_at_5": 0.50,
    "ndcg_at_5": 0.40,
    "mrr_at_5": 0.40,
    "relevance": 3.0,
    "correctness": 3.0,
    "faithfulness": 3.0,
    "coverage": 3.0,
    "groundedness": 3.0,
}

low_score = runs[
    runs.apply(
        lambda row: (
            row["metric_name"] in manual_cutoffs
            and row["metric_value"] < manual_cutoffs[row["metric_name"]]
        ),
        axis=1,
    )
].copy()

low_score.sort_values("metric_value").head(50)

## Inspect One Run

In [ ]:
# Pick an eval run id from `attention`, `low_score`, or `runs`.
eval_run_id = str(attention.iloc[0]["id"] if len(attention) else runs.iloc[0]["id"])
eval_run_id

In [ ]:
samples_sql = text("""
select
    sample_idx,
    sample_id,
    input,
    output,
    reference,
    detail
from eval_samples
where eval_run_id = :eval_run_id
order by sample_idx
""")

samples = pd.read_sql(samples_sql, engine, params={"eval_run_id": eval_run_id})
samples.head(20)

## Task-Specific Views

In [ ]:
def detail_value(detail, key, default=None):
    if isinstance(detail, dict):
        return detail.get(key, default)
    return default


run_row = runs.loc[runs["id"].astype(str) == eval_run_id].iloc[0]
task = run_row["task"]

if task == "retrieval":
    view = samples.assign(
        retrieved_ids=samples["detail"].map(lambda d: detail_value(d, "retrieved_ids", [])),
        relevance=samples["detail"].map(lambda d: detail_value(d, "relevance", {})),
    )
elif task == "code":
    view = samples.assign(
        passed=samples["detail"].map(lambda d: detail_value(d, "passed")),
        exit_code=samples["detail"].map(lambda d: detail_value(d, "exit_code")),
        stderr=samples["detail"].map(lambda d: detail_value(d, "stderr", "")),
    )
else:
    view = samples.assign(
        output_length=samples["output"].fillna("").str.len(),
        reference_length=samples["reference"].fillna("").str.len(),
        has_rag_context=samples["detail"].map(lambda d: bool(detail_value(d, "rag_context", ""))),
    )

view.head(30)

## Failure Categories

Use this table as the human review output. Keep categories stable enough that repeated runs can be compared.

In [ ]:
review = view[["sample_idx", "sample_id", "input", "output", "reference", "detail"]].copy()
review["failure_category"] = "unclassified"
review["operator_note"] = ""

review.head(20)

Suggested categories:

- `retrieval_no_hit`
- `retrieval_wrong_source`
- `prompt_missing_context`
- `generation_hallucination`
- `generation_incomplete`
- `format_or_contract_error`
- `code_runtime_error`
- `eval_dataset_issue`
- `metric_or_judge_issue`
- `infrastructure_error`

In [ ]:
category_summary = (
    review.groupby("failure_category", dropna=False)
    .size()
    .reset_index(name="samples")
    .sort_values("samples", ascending=False)
)
category_summary

## Export Candidate Dataset Rows

In [ ]:
candidate_rows = review[review["failure_category"] != "unclassified"].copy()
export_path = exports_dir / f"failure_candidates_{eval_run_id}.jsonl"
candidate_rows.to_json(export_path, orient="records", lines=True, force_ascii=False)
export_path